# Style2Fit — Step 4: Demo
Full pipeline demo with before/after comparison.

**Before running:** upload `llm_adapter.zip` and `sdxl_lora.zip` from previous steps.

**Runtime:** A100 GPU.

In [ ]:
!pip install transformers peft bitsandbytes diffusers accelerate gradio -q

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_...'  # your HuggingFace token

In [ ]:
# Upload and unzip the adapter weights
from google.colab import files
print('Upload llm_adapter.zip and sdxl_lora.zip')
files.upload()

In [ ]:
!unzip -q llm_adapter.zip
!unzip -q sdxl_lora.zip
!ls llm_adapter/ sdxl_lora/

In [ ]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from diffusers import StableDiffusionXLPipeline

BASE_LLM = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
BASE_SDXL = 'stabilityai/stable-diffusion-xl-base-1.0'

SYSTEM_PROMPT = """You are Style2Fit, a personal stylist assistant.
When someone describes their situation in casual language, you generate a complete,
coherent outfit recommendation in structured format.

Always respond with exactly:
Top: ...
Bottom: ...
Shoes: ...
Outerwear: ...
Accessories: ...
Aesthetic: ...
Explanation: ..."""

In [ ]:
# Load fine-tuned LLM
print('Loading LLM...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained('llm_adapter')
base = AutoModelForCausalLM.from_pretrained(BASE_LLM, quantization_config=bnb,
                                             device_map='auto', token=os.environ['HF_TOKEN'])
llm = PeftModel.from_pretrained(base, 'llm_adapter')
llm.eval()
print('LLM ready.')

In [ ]:
# Load fine-tuned SDXL
print('Loading SDXL...')
sdxl = StableDiffusionXLPipeline.from_pretrained(
    BASE_SDXL, torch_dtype=torch.bfloat16
).to('cuda')
sdxl.load_lora_weights('sdxl_lora')
sdxl.enable_attention_slicing()
print('SDXL ready.')

In [ ]:
def generate_outfit_plan(situation, aesthetic=None):
    user_content = situation
    if aesthetic:
        user_content += f'\naesthetic: {aesthetic}'
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_content},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).to(llm.device)
    with torch.no_grad():
        output = llm.generate(
            input_ids, max_new_tokens=300, temperature=0.7,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True)


def plan_to_image_prompt(plan_text):
    def extract(field):
        m = re.search(rf'{field}:\s*(.+?)(?:\n|$)', plan_text, re.IGNORECASE)
        return m.group(1).strip() if m else ''
    top      = extract('Top')
    bottom   = extract('Bottom')
    shoes    = extract('Shoes')
    outer    = extract('Outerwear')
    aesthetic = extract('Aesthetic')
    pieces = [top, bottom, shoes]
    if outer and outer.lower() not in ('none', 'none needed', 'n/a'):
        pieces.append(outer)
    outfit_desc = ', '.join(p for p in pieces if p)
    return (
        f'Full body fashion editorial photo of a person wearing {outfit_desc}. '
        f'{aesthetic} aesthetic. Soft natural lighting, clean background, '
        'professional fashion photography, sharp focus, high resolution.'
    )


def run_pipeline(situation, aesthetic=None):
    plan = generate_outfit_plan(situation, aesthetic)
    prompt = plan_to_image_prompt(plan)
    image = sdxl(
        prompt=prompt,
        negative_prompt='cropped, bad anatomy, blurry, low quality, cartoon',
        num_inference_steps=30, guidance_scale=7.5,
        height=1024, width=768,
    ).images[0]
    return plan, image


print('Pipeline functions ready.')

In [ ]:
# Test it
import matplotlib.pyplot as plt

situation = 'i have a coffee date tmrw what do i wear'
plan, image = run_pipeline(situation)

print(f'Situation: "{situation}"')
print('\nOutfit Plan:')
print(plan)

plt.figure(figsize=(5, 8))
plt.imshow(image)
plt.axis('off')
plt.title(situation, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Launch Gradio demo with public URL (for pitch day)
import gradio as gr

AESTHETICS = [
    'infer from situation', 'clean girl', 'dark academia', 'old money',
    'streetwear', 'soft girl', 'indie', 'minimalist', 'boho', 'preppy', 'y2k',
]

def gradio_fn(situation, aesthetic):
    if not situation.strip():
        return 'Please enter a situation.', None
    aes = None if aesthetic == 'infer from situation' else aesthetic
    plan, image = run_pipeline(situation, aes)
    formatted = plan.replace('\n', '\n\n')
    return formatted, image

with gr.Blocks(title='Style2Fit', theme=gr.themes.Soft(primary_hue='rose')) as demo:
    gr.Markdown('# Style2Fit ✨\n### Describe your situation. Get a real outfit. See it on a person.')
    with gr.Row():
        with gr.Column():
            situation_box = gr.Textbox(
                label='What\'s the situation?',
                placeholder='i have a coffee date tmrw what do i wear',
                lines=2,
            )
            aesthetic_box = gr.Dropdown(choices=AESTHETICS, value='infer from situation', label='Aesthetic (optional)')
            btn = gr.Button('Generate Outfit', variant='primary')
            gr.Examples(
                examples=[
                    ['i have a coffee date tmrw what do i wear', 'infer from situation'],
                    ['first day at my internship, business casual', 'infer from situation'],
                    ['concert this weekend, indie/alt vibe', 'indie'],
                    ['birthday dinner at a nice restaurant', 'old money'],
                    ['omg i have a presentation today help', 'infer from situation'],
                ],
                inputs=[situation_box, aesthetic_box],
            )
        with gr.Column():
            plan_out = gr.Textbox(label='Outfit Plan', lines=10)
            image_out = gr.Image(label='Outfit Visual', type='pil')
    btn.click(gradio_fn, inputs=[situation_box, aesthetic_box], outputs=[plan_out, image_out])
    situation_box.submit(gradio_fn, inputs=[situation_box, aesthetic_box], outputs=[plan_out, image_out])

demo.launch(share=True)  # share=True gives a public URL